# `dit` — Exact Discrete Information Theory

> **Docs**: https://dit.readthedocs.io/

**`dit`** computes information-theoretic measures from **explicit probability distributions** over discrete outcomes.
Because it works with the true probability mass function (PMF) rather than estimating from samples, all results are exact (up to floating-point).

### What we compute
| Measure | Symbol | Interpretation |
|---------|--------|---------------|
| Joint Shannon entropy | $H(X_0,X_1,X_2)$ | Total uncertainty of the system |
| Total Correlation | $TC$ | Shared information across all variables |
| Co-information | $CI$ (= $\Omega$ for $n=3$) | Signed: positive = redundancy, negative = synergy |

### Datasets
* **Artificial** — controlled redundant and synergistic triplets (sanity check).
* **Real** — UCI Wine dataset (top-3 features by MI with class label).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import random
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.datasets import load_wine
from sklearn.feature_selection import mutual_info_classif

import dit
from dit import Distribution
from dit.shannon import entropy
from dit.multivariate import total_correlation, coinformation

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print("Imports OK.")


## 1. Artificial Data

Two three-variable Gaussian systems with known ground-truth structure:

| Dataset | Construction | Expected $\Omega$ |
|---------|-------------|-------------------|
| `data_red` | All three are noisy copies of one source | $> 0$ (redundancy) |
| `data_syn` | $X_2 = X_0 + X_1 + \varepsilon$ | $< 0$ (synergy) |


In [ ]:
n_features, n_samples = 3, 500
eta = 0.1  # noise level

# Redundant: all three are copies of one source
source = random.normal(0, 1, n_samples)
data_red = np.array([
    source,
    source + random.normal(0, 1, n_samples) * eta,
    source + random.normal(0, 1, n_samples) * eta,
])

# Synergistic: X2 = X0 + X1 + noise
s0 = random.normal(0, 1, n_samples)
s1 = random.normal(0, 1, n_samples)
data_syn = np.array([
    s0 + random.normal(0, 1, n_samples) * eta,
    s1 + random.normal(0, 1, n_samples) * eta,
    s0 + s1 + random.normal(0, 1, n_samples) * eta,
])

print("data_red shape:", data_red.shape)
print("data_syn shape:", data_syn.shape)


## 2. Discretisation

`dit` requires **discrete** integer-valued inputs.
We bin each continuous variable independently into $k = 10$ equal-width bins.


In [ ]:
def discretize(data, n_bins=10):
    """Bin continuous (n_features, n_samples) array into integer bins."""
    est = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform')
    return est.fit_transform(data.T).astype(int)  # returns (n_samples, n_features)

N_BINS = 10
red_disc = discretize(data_red, N_BINS)
syn_disc = discretize(data_syn, N_BINS)
print("Discretised shapes:", red_disc.shape, syn_disc.shape)


## 3. Build `dit` Distributions

We count how often each outcome tuple $(x_0, x_1, x_2)$ appears and normalise to get the joint PMF.


In [ ]:
def create_distribution(disc_data):
    """Build a dit.Distribution from a discretised (n_samples, n_features) array."""
    tuples  = [tuple(row) for row in disc_data]
    values, counts = np.unique(tuples, axis=0, return_counts=True)
    probs    = counts / counts.sum()
    outcomes = [tuple(map(str, v)) for v in values]
    return Distribution(outcomes, probs)

red_dist = create_distribution(red_disc)
syn_dist = create_distribution(syn_disc)
print("Distinct outcomes — Redundant:", len(red_dist.outcomes),
      "  Synergistic:", len(syn_dist.outcomes))


## 4. Compute Measures

In [ ]:
# Joint entropy
H_red = entropy(red_dist)
H_syn = entropy(syn_dist)
print(f"Joint entropy H(X0,X1,X2)")
print(f"  Redundant:    {H_red:.4f} bits")
print(f"  Synergistic:  {H_syn:.4f} bits")

# Total Correlation
TC_red = total_correlation(red_dist)
TC_syn = total_correlation(syn_dist)
print(f"\nTotal Correlation TC(X0,X1,X2)")
print(f"  Redundant:    {TC_red:.4f} bits  ← high (variables share variance)")
print(f"  Synergistic:  {TC_syn:.4f} bits")

# Co-information (= O-information for n=3)
CI_red = coinformation(red_dist)
CI_syn = coinformation(syn_dist)
print(f"\nCo-information CI(X0;X1;X2)  [= Ω for n=3]")
print(f"  Redundant:    {CI_red:+.4f} bits   ← POSITIVE ✓ redundancy confirmed")
print(f"  Synergistic:  {CI_syn:+.4f} bits   ← NEGATIVE ✓ synergy confirmed")


## 5. Visualisation — Artificial Data

In [ ]:
labels = ['$H(X_0,X_1,X_2)$', '$TC$', '$CI$ / $\\Omega$']
vals_r = [H_red, TC_red, CI_red]
vals_s = [H_syn, TC_syn, CI_syn]

x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, vals_r, w, label='Redundant',   color=sns.color_palette()[0])
ax.bar(x + w/2, vals_s, w, label='Synergistic', color=sns.color_palette()[2])
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Value (bits)')
ax.set_title('dit: entropy, TC, and co-information\nfor redundant vs synergistic triplets',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Real Data — UCI Wine Dataset

We rank features by their MI with the class label, select the top 3, discretise them, and compute the same measures.


In [ ]:
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
feature_names = list(wine.feature_names)

# Rank features by MI with class label
mi_scores = mutual_info_classif(X_wine, y_wine, random_state=SEED)
order = np.argsort(mi_scores)[::-1]
top3 = [feature_names[i] for i in order[:3]]
print("Top-3 features:", top3)

# Extract, discretise, build distribution
X_top3 = X_wine[:, order[:3]]
disc_wine = discretize(X_top3.T, n_bins=8)
wine_dist  = create_distribution(disc_wine)

# Measures
H_w  = entropy(wine_dist)
TC_w = total_correlation(wine_dist)
CI_w = coinformation(wine_dist)

print(f"\nWine top-3 features: {top3}")
print(f"  H  = {H_w:.4f} bits")
print(f"  TC = {TC_w:.4f} bits")
print(f"  CI = {CI_w:+.4f} bits  ({'redundancy-dominated' if CI_w > 0 else 'synergy-dominated'})")


In [ ]:
labels_w = ['$H$', '$TC$', '$CI$']
vals_w   = [H_w, TC_w, CI_w]
colors   = [sns.color_palette()[i] for i in range(3)]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels_w, vals_w, color=colors)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_ylabel('Value (bits)')
ax.set_title('dit: information measures on Wine top-3 features', fontweight='bold')
for bar, val in zip(bars, vals_w):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:+.3f}',
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()
